# Analyze Track Library to Produce Chromagram Vectors

For each track in the given track library path list, this notebook produces a chromagram vector for each track, per time step within each track.

Each chromagram vector covers the entire piano keyboard, per note. Stated more technically, each chromagram vector covers each note within the scientific pitch notation range C0 through B8. 

## Load useful libraries

In [ ]:
import librosa
import multiprocessing
import pandas as pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F

In [ ]:
from chromagram_functions import process_octave

## User settings

In [ ]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 300
spark_memory = '70G'

octave_min_inclusive = 0
octave_max_inclusive = 8

# location of tracks to analyze for building the track feature database
path_library_parquet = '../database/music/output/playlist.parquet'

soundfile_root_directory = '/media/emily/DJ_Backup/rekordbox_bak'

## Read a dataframe containing the file locations of each track in the library

In [ ]:
pdf_library_paths = pd.read_parquet(path_library_parquet)

In [ ]:
pdf_library_paths.head(5)

In [ ]:
import glob
glob_search_root_directory = soundfile_root_directory

pattern = glob_search_root_directory + '/**/*.mp3'
recursive_files = glob.glob(pattern, recursive=True)

pattern = glob_search_root_directory + '/**/*.MP3'
recursive_files.extend(glob.glob(pattern, recursive=True))

recursive_files[-5:]

In [ ]:
from tinytag import TinyTag

true_path_list = []
for track_file in recursive_files:
    tag = TinyTag.get(track_file)

    df_temp = pdf_library_paths[(pdf_library_paths['artist'] == tag.artist) & (pdf_library_paths['name'] == tag.title)]
    if len(df_temp.index) > 0:
        true_path_list.append(
            {
                'true_path' : track_file,
                'artist' : tag.artist,
                'name' : tag.title,
            }
        )

df_true_path = pd.DataFrame(true_path_list)

In [ ]:
df_true_path = (
    pd.merge(
        df_true_path,
        pdf_library_paths,
        on = ['artist', 'name'],
        how = 'left',
    )
    .drop(columns = ['path'])
)

In [ ]:
df_true_path

In [ ]:
len(df_true_path.index)

In [ ]:
len(df_true_path.dropna().index)

In [ ]:
df_true_path.to_parquet(output_directory + '/df_true_path.parquet')

## Define function enabling parallelized track analysis

In [ ]:
def process_song_from_song_library(filename, track_id, sampling_rate, hop_length, octave_min_inclusive, octave_max_inclusive):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now, neither efficient nor too costly

    # there is a file not found error I need to debug later
    try:
        y, sr = librosa.load(filename, sr = sampling_rate, mono = True)
    except:
        print()
        print(filename)
        print()
        return None
    
    y_harmonic, y_percussive = librosa.effects.hpss(y)
    results_list = []
    for octave_number in range(octave_min_inclusive, octave_max_inclusive + 1):
        chromagram = process_octave(y_harmonic, octave_number, sr = sampling_rate, hop_length = hop_length)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    df_all_octaves['id'] = track_id
    return df_all_octaves

## Mathematically process track library to extract chromagram features

We obtain a chromagram vector that contains a separate value for every note on the piano, per time step.

Runs in parallel.

NOTE:  Using the 'capture' function suppresses warnings. Later I need to figure out why these warnings happen (they occur for an extremely small minority of tracks so I am ignoring this matter for now).

In [ ]:
args_list = []
for i, row in df_true_path.iterrows():
    args_list.append((row['true_path'], row['id'], sampling_rate, hop_length, octave_min_inclusive, octave_max_inclusive))

with multiprocessing.Pool(processes = 50) as pool:
    results = pool.starmap(process_song_from_song_library, args_list)

In [ ]:
len(results)

## Assemble track library feature vectors into a DataFrame

In [ ]:
pdf_library = pd.concat(results, ignore_index = True)

In [ ]:
pdf_library.head(3)

## QA

In [ ]:
len(pdf_library.index)

In [ ]:
len(pdf_library.dropna().index)

In [ ]:
len(pdf_library['id'].unique())

## Remove rows with NaN values

I don't expect there to be any, and if the QA procedure above finds any, we need to figure out why.

In [ ]:
pdf_library.dropna(inplace = True)

## Identify the column names for the notes

In [ ]:
pitch_columns = [x for x in pdf_library.columns if x != 'id']

## Initiate a Spark session

In [ ]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

## Convert the Pandas DF to a Spark DF and collapse value columns to a vector

In [ ]:
sdf_library = (
    spark
    .createDataFrame(pdf_library)
    .orderBy('id')
    .repartition('id')
    .withColumn('array_library', F.array(*pitch_columns))
    .select('array_library', 'id')
)

In [ ]:
sdf_library.show(5)

## Write the Spark DataFrame to a Parquet file

In [ ]:
path_library_output = output_directory + '/library_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library.write.mode('overwrite').parquet(path_library_output)

## Close the Spark session

In [ ]:
spark.stop()